# Download Arsenal

Ноутбук использует две готовые конфигурации: многомодельную `test_playbook_arsenal_router_mode.toml` для Router Mode и `test_playbook_arsenal_model_mode.toml` с одной моделью на сервер для Model Mode. Уже загруженные файлы повторно не скачиваются.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = next(
    directory
    for directory in (Path.cwd(), *Path.cwd().parents)
    if (directory / ".zemicomp").is_file() and (directory / "zemi").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from zemi.playbook import arsenal

## Загрузка

Следующие ячейки выполняют реальные сетевые загрузки и могут работать продолжительное время. Прогресс каждого llama-сервера и каждой модели отображается в output ячейки.

In [2]:
router_mode_arsenal = arsenal.download(
    "@comp/tests/playbook_arsenal/test_playbook_arsenal_router_mode.toml"
)

══════════════════════════════════════════════════════════════════════════════
ZEMI Playbook · загрузка Arsenal
Конфигурация: @comp/tests/playbook_arsenal/test_playbook_arsenal_router_mode.toml
══════════════════════════════════════════════════════════════════════════════
Найдено серверов: 2 · моделей: 4

──────────────────────────────────────────────────────────────────────────────
LLAMA-SERVER [1/2] · primary · llama:b9222
──────────────────────────────────────────────────────────────────────────────
llama.cpp b9222 уже загружен: @inst/_llamas/llama--b9222

  МОДЕЛЬ [1/4] · primary/qwen
  bartowski/Qwen_Qwen3.5-4B-GGUF/Qwen_Qwen3.5-4B-Q4_K_M.gguf
Модель уже загружена: @inst/_models/hf--bartowski--Qwen_Qwen3.5-4B-GGUF--Qwen_Qwen3.5-4B-Q4_K_M/Qwen_Qwen3.5-4B-Q4_K_M.gguf

  МОДЕЛЬ [2/4] · primary/smollm
  bartowski/SmolLM2-1.7B-Instruct-GGUF/SmolLM2-1.7B-Instruct-Q4_K_M.gguf
Модель уже загружена: @inst/_models/hf--bartowski--SmolLM2-1.7B-Instruct-GGUF--SmolLM2-1.7B-Instruct-Q4_K_M/SmolL

In [3]:
model_mode_arsenal = arsenal.download(
    "@comp/tests/playbook_arsenal/test_playbook_arsenal_model_mode.toml"
)

══════════════════════════════════════════════════════════════════════════════
ZEMI Playbook · загрузка Arsenal
Конфигурация: @comp/tests/playbook_arsenal/test_playbook_arsenal_model_mode.toml
══════════════════════════════════════════════════════════════════════════════
Найдено серверов: 2 · моделей: 2

──────────────────────────────────────────────────────────────────────────────
LLAMA-SERVER [1/2] · primary · llama:b9222
──────────────────────────────────────────────────────────────────────────────
llama.cpp b9222 уже загружен: @inst/_llamas/llama--b9222

  МОДЕЛЬ [1/2] · primary/qwen
  bartowski/Qwen_Qwen3.5-4B-GGUF/Qwen_Qwen3.5-4B-Q4_K_M.gguf
Модель уже загружена: @inst/_models/hf--bartowski--Qwen_Qwen3.5-4B-GGUF--Qwen_Qwen3.5-4B-Q4_K_M/Qwen_Qwen3.5-4B-Q4_K_M.gguf

──────────────────────────────────────────────────────────────────────────────
LLAMA-SERVER [2/2] · secondary · llama:b9222
──────────────────────────────────────────────────────────────────────────────
llama.cpp b9222 

# Begin и end playbook

Ниже приведены осмысленные варианты запуска и завершения. Каждая ячейка запускает или останавливает реальные процессы `llama-server`. Не запускайте несколько сценариев `begin_playbook` подряд без промежуточного `end_playbook(stop_arsenal_after_end=True)`, если в новом сценарии не включена предварительная остановка.

### Model Mode: чистый запуск

Рекомендуемый вариант для воспроизводимого запуска: сначала остановить возможные старые процессы Arsenal, затем запустить по одному серверу на модель и остановить их после playbook.

In [4]:
model_mode_arsenal.begin_playbook(
    stop_arsenal_before_begin=True,
    llama_router_mode=False,
)

# Здесь выполняются шаги playbook.

model_mode_arsenal.end_playbook(stop_arsenal_after_end=True)


══════════════════════════════════════════════════════════════════════════════
ZEMI Playbook · ОСТАНОВКА ARSENAL
Llama-серверов: 2
══════════════════════════════════════════════════════════════════════════════
[1/2] primary · 127.0.0.1:8080
    · не запущен
[2/2] secondary · 127.0.0.1:8081
    · не запущен
══════════════════════════════════════════════════════════════════════════════
✓ Arsenal остановлен
══════════════════════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════════════════════
ZEMI Playbook · ЗАПУСК ARSENAL · MODEL MODE
Llama-серверов: 2
══════════════════════════════════════════════════════════════════════════════

[1/2] Запускаю primary · 127.0.0.1:8080
    ✓ готов · PID 5068

[2/2] Запускаю secondary · 127.0.0.1:8081
    ✓ готов · PID 8688
══════════════════════════════════════════════════════════════════════════════
✓ Все llama-серверы готовы
═══════════════════════════════════════════════════════

### Model Mode: запуск без предварительной остановки

Используйте только когда известно, что настроенные порты свободны. Завершение с `False` намеренно оставляет серверы работающими для следующего playbook; последняя строка показывает явную последующую очистку.

In [5]:
model_mode_arsenal.begin_playbook(
    stop_arsenal_before_begin=False,
    llama_router_mode=False,
)

# Серверы остаются доступны после завершения playbook.
model_mode_arsenal.end_playbook(stop_arsenal_after_end=False)

# Выполните позже, когда серверы больше не нужны.
model_mode_arsenal.end_playbook(stop_arsenal_after_end=True)


══════════════════════════════════════════════════════════════════════════════
ZEMI Playbook · ЗАПУСК ARSENAL · MODEL MODE
Llama-серверов: 2
══════════════════════════════════════════════════════════════════════════════

[1/2] Запускаю primary · 127.0.0.1:8080
    ✓ готов · PID 14020

[2/2] Запускаю secondary · 127.0.0.1:8081
    ✓ готов · PID 4424
══════════════════════════════════════════════════════════════════════════════
✓ Все llama-серверы готовы
══════════════════════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════════════════════
ZEMI Playbook · ОСТАНОВКА ARSENAL
Llama-серверов: 2
══════════════════════════════════════════════════════════════════════════════
[1/2] primary · 127.0.0.1:8080
    ✓ остановлен · PID 14020
[2/2] secondary · 127.0.0.1:8081
    ✓ остановлен · PID 4424
══════════════════════════════════════════════════════════════════════════════
✓ Arsenal остановлен
═══════════════════════════════

## Router Mode: чистый запуск

Каждый сервер получает все свои модели через сгенерированный INI-пресет. Предварительная и завершающая остановка делают сценарий полностью изолированным.

In [6]:
router_mode_arsenal.begin_playbook(
    stop_arsenal_before_begin=True,
    llama_router_mode=True,
)


══════════════════════════════════════════════════════════════════════════════
ZEMI Playbook · ОСТАНОВКА ARSENAL
Llama-серверов: 2
══════════════════════════════════════════════════════════════════════════════
[1/2] primary · 127.0.0.1:8080
    · не запущен
[2/2] secondary · 127.0.0.1:8081
    · не запущен
══════════════════════════════════════════════════════════════════════════════
✓ Arsenal остановлен
══════════════════════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════════════════════
ZEMI Playbook · ЗАПУСК ARSENAL · ROUTER MODE
Llama-серверов: 2
══════════════════════════════════════════════════════════════════════════════
    Пресет primary: 2 моделей · C:\Users\Axoman\Documents\ZEMI\_tmp\zemi-arsenal-primary.ini

[1/2] Запускаю primary · 127.0.0.1:8080
    ✓ готов · PID 14092
    Пресет secondary: 2 моделей · C:\Users\Axoman\Documents\ZEMI\_tmp\zemi-arsenal-secondary.ini

[2/2] Запускаю secondary · 127.0.0

### Объектная модель запущенного Arsenal

После `begin_playbook` серверы уже готовы, а вся конфигурация доступна через `llamas → models → assistants`. На каждом уровне один объект можно получить по индексу, строковому имени или через точку; свойство `config` содержит соответствующую исходную TOML-таблицу.

In [7]:
# Llama-сервер: индекс, имя и точечная запись.
primary_by_index = router_mode_arsenal.llamas[0]
primary_by_name = router_mode_arsenal.llamas["primary"]
primary_by_dot = router_mode_arsenal.llamas.primary
assert primary_by_index is primary_by_name is primary_by_dot

# Модель: те же три способа доступа.
qwen_by_index = primary_by_dot.models[0]
qwen_by_name = primary_by_dot.models["qwen"]
qwen_by_dot = primary_by_dot.models.qwen
assert qwen_by_index is qwen_by_name is qwen_by_dot

# Ассистент: те же три способа доступа.
assistant_by_index = qwen_by_dot.assistants[0]
assistant_by_name = qwen_by_dot.assistants["assistant"]
assistant_by_dot = qwen_by_dot.assistants.assistant
assert assistant_by_index is assistant_by_name is assistant_by_dot

# Коллекции сохраняют порядок и поддерживают отрицательные индексы.
assert router_mode_arsenal.llamas[-1].name == "secondary"
assert primary_by_dot.models[-1].name == "smollm"

{
    "llama_names": list(router_mode_arsenal.llamas.keys()),
    "model_names": list(primary_by_dot.models.keys()),
    "assistant_names": list(qwen_by_dot.assistants.keys()),
    "llama_config": primary_by_dot.config,
    "model_config": qwen_by_dot.config,
    "assistant_config": assistant_by_dot.config,
}

{'llama_names': ['primary', 'secondary'],
 'model_names': ['qwen', 'smollm'],
 'assistant_names': ['assistant', 'json_converter'],
 'llama_config': {'name': 'primary',
  'llama_build': 'llama:b9222',
  'host': '127.0.0.1',
  'port': 8080,
  'startup_timeout': 120.0,
  'models': {'qwen': {'name': 'qwen',
    'source': 'hf',
    'owner': 'bartowski',
    'repository': 'Qwen_Qwen3.5-4B-GGUF',
    'filename': 'Qwen_Qwen3.5-4B-Q4_K_M.gguf',
    'alias': 'qwen3.5-4b',
    'ctx_size': 8192,
    'threads': 8,
    'threads_batch': 8,
    'reasoning': 'off',
    'assistants': {'assistant': {'name': 'assistant',
      'prefix': 'Ты — универсальный ассистент проекта ZEMI.\n\nОтвечай на русском языке, если пользователь не попросил иначе. Формулируй ответы точно и по существу. Если данных недостаточно, явно укажи, чего именно не хватает.\n'},
     'json_converter': {'name': 'json_converter',
      'prefix': 'Преобразуй входные данные в корректный JSON.\n\nВозвращай только JSON без Markdown-ограждени

### Завершение playbook

После демонстрации объектной модели останавливаем все серверы из конфигурации.

In [8]:
router_mode_arsenal.end_playbook(stop_arsenal_after_end=True)


══════════════════════════════════════════════════════════════════════════════
ZEMI Playbook · ОСТАНОВКА ARSENAL
Llama-серверов: 2
══════════════════════════════════════════════════════════════════════════════
[1/2] primary · 127.0.0.1:8080
    ✓ остановлен · PID 14092
[2/2] secondary · 127.0.0.1:8081
    ✓ остановлен · PID 15288
══════════════════════════════════════════════════════════════════════════════
✓ Arsenal остановлен
══════════════════════════════════════════════════════════════════════════════


## Router Mode: запуск на заведомо свободных портах

Вариант без предварительной остановки полезен, когда состояние окружения контролируется снаружи. Серверы останавливаются после playbook.

In [9]:
router_mode_arsenal.begin_playbook(
    stop_arsenal_before_begin=False,
    llama_router_mode=True,
)

# Здесь выполняются шаги playbook с выбором модели в запросах.

router_mode_arsenal.end_playbook(stop_arsenal_after_end=True)


══════════════════════════════════════════════════════════════════════════════
ZEMI Playbook · ЗАПУСК ARSENAL · ROUTER MODE
Llama-серверов: 2
══════════════════════════════════════════════════════════════════════════════
    Пресет primary: 2 моделей · C:\Users\Axoman\Documents\ZEMI\_tmp\zemi-arsenal-primary.ini

[1/2] Запускаю primary · 127.0.0.1:8080
    ✓ готов · PID 16232
    Пресет secondary: 2 моделей · C:\Users\Axoman\Documents\ZEMI\_tmp\zemi-arsenal-secondary.ini

[2/2] Запускаю secondary · 127.0.0.1:8081
    ✓ готов · PID 18004
══════════════════════════════════════════════════════════════════════════════
✓ Все llama-серверы готовы
══════════════════════════════════════════════════════════════════════════════

══════════════════════════════════════════════════════════════════════════════
ZEMI Playbook · ОСТАНОВКА ARSENAL
Llama-серверов: 2
══════════════════════════════════════════════════════════════════════════════
[1/2] primary · 127.0.0.1:8080
    ✓ остановлен · PID 16232


## Проверка ограничения Model Mode

Исходный Arsenal содержит несколько моделей на сервер. Поэтому попытка запустить его без Router Mode ожидаемо завершается `ValueError` до запуска первого сервера.

In [10]:
try:
    router_mode_arsenal.begin_playbook(
        stop_arsenal_before_begin=False,
        llama_router_mode=False,
    )
except ValueError as error:
    print(f"Ожидаемая ошибка конфигурации: {error}")


══════════════════════════════════════════════════════════════════════════════
ZEMI Playbook · ЗАПУСК ARSENAL · MODEL MODE
Llama-серверов: 2
══════════════════════════════════════════════════════════════════════════════
✗ Проверка конфигурации не пройдена: primary (2 моделей), secondary (2 моделей)
══════════════════════════════════════════════════════════════════════════════
Ожидаемая ошибка конфигурации: Без Router Mode каждый llama-сервер должен содержать ровно одну модель. Нарушение: primary (2 моделей), secondary (2 моделей)
